# Scénario 11f : ce qu'apporte `PAY_1_corrigee`, à réglages fixés

Journal : [`tableau_resultats.md`](tableau_resultats.md).

**Question** : ajoutée à côté de `PAY_1`, la colonne `PAY_1_corrigee` améliore-t-elle la prédiction du défaut (`dpnm`), **toutes choses égales par ailleurs** ?

**Pourquoi ce notebook** : le `11e` comparait son modèle à `essai_jeu_de_base`, mais plusieurs choses changeaient à la fois : la colonne ajoutée, les codifications 1 d'avril à août corrigées, le plafond au logarithme, et surtout **les réglages**, choisis à nouveau par la boucle. Ici, **une seule chose change** entre les deux modèles comparés : la présence de `PAY_1_corrigee`.

**Ce qui est identique dans les deux modèles** :
- les données : les 30 000 clients de `data/creditcard_pret_ingestion.csv`, codifications 1 d'avril à août corrigées par le `10` (sans la règle fixe des retards), comme dans le `11e` ;
- les encodages du `11e` : `PAY_n` en nombres mis à l'échelle, paiements et plafond au logarithme, factures au logarithme signé, `SEX`, `EDUCATION`, `MARRIAGE` en une colonne par catégorie ;
- les **réglages, fixés sans boucle**, et les graines ;
- les 5 plis (`StratifiedKFold`, `random_state=42`) et le seuil de décision (règle marginale d'`essai_jeu_de_base` : au plus 1 bon client signalé pour 1 défaut, à la marge, choisi dans des plis internes).

**Les deux modèles comparés** : **sans** `PAY_1_corrigee` (23 variables) et **avec** (24 variables, `PAY_1` d'origine gardée).

**Deux jeux de réglages**, pour vérifier que la conclusion ne tient pas à un réglage particulier :
- **réglages A** : ceux retenus par `essai_jeu_de_base` (variante complète, sans la colonne) ;
- **réglages B** : ceux retenus par le `11e` (variante complète, avec la colonne).

`PAY_1_corrigee` est lue dans `sorties/pay1_corrigee.csv`, écrite par le `11e` (même construction : codification d'août pour les 1 sains en août ; -1 si le MLP et CatBoost du `11c` annoncent une sortie du retard, 2 sinon). ⚠️ Le test du ML est inclus dans les 30 000 clients, comme dans `essai_jeu_de_base` : cet essai éclaire la question, il ne choisit pas le modèle final.

## 1. Données : codifications 1 d'avril à août corrigées, `PAY_1_corrigee` lue dans le `11e`

In [ ]:
import time
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import Markdown, display

current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())
original = pd.read_csv(root_dir / "data" / "creditcard_pret_ingestion.csv")
assert original["ID"].is_unique and not original.isna().any().any()
y = original["dpnm"].to_numpy()
PAY = [f"PAY_{n}" for n in range(1, 7)]
avant = original[PAY].copy()

def saine_anterieure(ligne, n):
    for k in range(n + 1, 7):
        if ligne[f"PAY_{k}"] <= 0:
            return ligne[f"PAY_{k}"]
    return None

# Correctifs du 10, des mois les plus anciens aux plus récents, sans la règle des retards du 09 (identiques au 11e)
for n in range(6, 1, -1):
    for k in original.index[(original[f"PAY_{n}"] == 1) & (original[f"BILL_AMT{n - 1}"] <= 0)]:
        v = saine_anterieure(original.loc[k], n)
        if v is not None:
            original.at[k, f"PAY_{n}"] = v
for n in range(5, 1, -1):
    regle = (original[f"PAY_{n}"] == 1) & (original[f"BILL_AMT{n}"] <= 0) & (original[f"PAY_{n + 1}"] <= 0)
    original.loc[regle, f"PAY_{n}"] = original.loc[regle, f"PAY_{n + 1}"]
DECISIONS = {6783: {4: 0, 3: 0, 2: 0}, 11498: {4: 2, 3: 2, 2: 2}, 892: {3: 0, 2: 0}, 1967: {3: 0, 2: 0},
             5645: {2: 0}, 19195: {2: 2}, 25176: {2: 2}}
for id_client, valeurs in DECISIONS.items():
    ligne = original.index[original["ID"] == id_client]
    for n, v in valeurs.items():
        original.loc[ligne, f"PAY_{n}"] = v
assert int((original[PAY[1:]] == 1).sum().sum()) == 0 and (original["PAY_1"] == avant["PAY_1"]).all()

colonne = pd.read_csv(Path.cwd() / "sorties" / "pay1_corrigee.csv")
assert (colonne["ID"].to_numpy() == original["ID"].to_numpy()).all() and (colonne["PAY_1"].to_numpy() == original["PAY_1"].to_numpy()).all()
original["PAY_1_corrigee"] = colonne["PAY_1_corrigee"].to_numpy()
assert not (original["PAY_1_corrigee"] == 1).any()
print(f"Clients : {len(original)} ; défauts : {y.sum()} ({y.mean() * 100:.2f} %)")
print(f"Codifications d'avril à août corrigées : {int((original[PAY] != avant).sum().sum())} (clients : {int((original[PAY] != avant).any(axis=1).sum())})")
print(f"PAY_1_corrigee différente de PAY_1 : {int((original['PAY_1_corrigee'] != original['PAY_1']).sum())} clients (les PAY_1 = 1)")

## 2. Encodages, réglages fixés, plis et seuil de décision

Les réglages sont fixés : aucune boucle d'optimisation, pour que les deux modèles comparés soient strictement les mêmes, à la colonne près. Les autres paramètres sont ceux d'`essai_jeu_de_base` (poids des défauts, graines, arrêt anticipé du MLP).

In [ ]:
from catboost import CatBoostClassifier
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, make_scorer, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, TunedThresholdClassifierCV
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

TYPES = {
    "num": ["AGE"] + PAY,
    "log": [f"PAY_AMT{n}" for n in range(1, 7)] + ["LIMIT_BAL"],
    "log_signe": [f"BILL_AMT{n}" for n in range(1, 7)],
    "nom": ["SEX", "EDUCATION", "MARRIAGE"],
}
VARIANTES = {"Sans PAY_1_corrigee": [], "Avec PAY_1_corrigee": ["PAY_1_corrigee"]}   # PAY_1_corrigee encodée comme les autres PAY_n
COLONNES = {v: [c for cols in TYPES.values() for c in cols] + plus for v, plus in VARIANTES.items()}
assert sorted(COLONNES["Avec PAY_1_corrigee"]) == sorted(c for c in original.columns if c not in ("ID", "dpnm"))

TRANSFORMATIONS = {
    "num": lambda: StandardScaler(),
    "log": lambda: make_pipeline(FunctionTransformer(np.log1p, feature_names_out="one-to-one"), StandardScaler()),
    "log_signe": lambda: make_pipeline(FunctionTransformer(np.arcsinh, feature_names_out="one-to-one"), StandardScaler()),
    "nom": lambda: OneHotEncoder(sparse_output=False, handle_unknown="ignore"),
}

def preprocesseur(variante):
    types = {t: list(cols) for t, cols in TYPES.items()}
    types["num"] += VARIANTES[variante]
    return ColumnTransformer([(t, TRANSFORMATIONS[t](), cols) for t, cols in types.items()])

scale_pos = (y == 0).sum() / (y == 1).sum()
REGLAGES = {
    "A (essai_jeu_de_base)": {
        "LogisticRegression": {"C": 10.0},
        "MLPClassifier": {"alpha": 0.1, "hidden_layer_sizes": (64, 32)},
        "RandomForest": {"max_depth": 8, "min_samples_leaf": 50, "n_estimators": 300},
        "CatBoost": {"depth": 4, "iterations": 667, "l2_leaf_reg": 33.8, "learning_rate": 0.02},
    },
    "B (11e)": {
        "LogisticRegression": {"C": 0.00111},
        "MLPClassifier": {"alpha": 0.1, "hidden_layer_sizes": (32, 16)},
        "RandomForest": {"max_depth": 6, "min_samples_leaf": 20, "n_estimators": 300},
        "CatBoost": {"depth": 4, "iterations": 667, "l2_leaf_reg": 22.5, "learning_rate": 0.03},
    },
}

def modele(nom, variante, params):
    prep = preprocesseur(variante)
    if nom == "LogisticRegression":
        m = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42, **params)
    elif nom == "MLPClassifier":
        m = MLPClassifier(max_iter=500, random_state=42, early_stopping=True, **params)
    elif nom == "RandomForest":
        m = RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1, **params)
    else:
        m = CatBoostClassifier(scale_pos_weight=scale_pos, random_state=42, verbose=0, thread_count=-1, allow_writing_files=False, **params)
    return make_pipeline(prep, m)

plis = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(original, y))
cv_interne = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
BONS_CLIENTS_PAR_DEFAUT = 1

def gain_net(y_vrai, y_pred):
    y_vrai, y_pred = np.asarray(y_vrai), np.asarray(y_pred)
    return ((y_pred & (y_vrai == 1)).sum() - BONS_CLIENTS_PAR_DEFAUT * (y_pred & (y_vrai == 0)).sum()) / len(y_vrai)

RAPPELS = [0.50, 0.60, 0.70, 0.80]

def seuil_pour_rappel(y_vrai, proba, rappel):
    p = np.sort(proba[np.asarray(y_vrai) == 1])[::-1]
    return p[int(np.ceil(rappel * len(p))) - 1]

print("Variantes :", {v: len(c) for v, c in COLONNES.items()}, "; réglages :", list(REGLAGES))

## 3. Les deux modèles, pli par pli

Pour chaque jeu de réglages, chaque modèle et chaque pli : le modèle **sans** et le modèle **avec** `PAY_1_corrigee` sont entraînés sur les mêmes 4 plis, avec les mêmes réglages et la même graine, puis jugés sur le même pli de validation. Une ligne par modèle s'affiche pendant le calcul.

In [ ]:
mesures, modeles_plis = {}, {}
for reglage, par_modele in REGLAGES.items():
    for nom, params in par_modele.items():
        debut = time.perf_counter()
        for variante in VARIANTES:
            X = original[COLONNES[variante]]
            m = {f"Précision au rappel de {r:.0%}": [] for r in RAPPELS}
            m.update({"Précision moyenne": [], "ROC AUC": [], "Rappel, règle marginale": [], "Précision, règle marginale": []})
            entraines = []
            for apprentissage, validation in plis:
                seuil_auto = TunedThresholdClassifierCV(modele(nom, variante, params), scoring=make_scorer(gain_net), cv=cv_interne, n_jobs=-1)
                seuil_auto.fit(X.iloc[apprentissage], y[apprentissage])
                p = seuil_auto.predict_proba(X.iloc[validation])[:, 1]
                decision = seuil_auto.predict(X.iloc[validation]).astype(bool)
                y_val = y[validation]
                for r in RAPPELS:
                    m[f"Précision au rappel de {r:.0%}"].append(y_val[p >= seuil_pour_rappel(y_val, p, r)].mean())
                m["Précision moyenne"].append(average_precision_score(y_val, p))
                m["ROC AUC"].append(roc_auc_score(y_val, p))
                m["Rappel, règle marginale"].append(recall_score(y_val, decision))
                m["Précision, règle marginale"].append(precision_score(y_val, decision, zero_division=0))
                entraines.append(seuil_auto)
            mesures[(reglage, nom, variante)] = {k: np.array(v) for k, v in m.items()}
            modeles_plis[(reglage, nom, variante)] = entraines
        print(f"Réglages {reglage}, {nom} : fait ({round(time.perf_counter() - debut)} s)", flush=True)

## 4. Ce qu'apporte `PAY_1_corrigee`

Pour chaque jeu de réglages, chaque modèle et chaque mesure : le score sans et avec la colonne (moyenne des 5 plis), le gain pli par pli (moyenne ± écart-type), les plis gagnés et le verdict : « gain réel » si le gain moyen dépasse l'écart-type des gains, « perte » s'il est en dessous de moins cet écart-type, sinon « dans le bruit ».

In [ ]:
def verdict(gains):
    moyenne, ecart_type = np.mean(gains), np.std(gains)
    return "✅ gain réel" if moyenne > ecart_type else ("❌ perte" if moyenne < -ecart_type else "➖ dans le bruit")

lignes = []
for reglage, par_modele in REGLAGES.items():
    for nom in par_modele:
        sans, avec = mesures[(reglage, nom, "Sans PAY_1_corrigee")], mesures[(reglage, nom, "Avec PAY_1_corrigee")]
        for mesure in sans:
            gains = avec[mesure] - sans[mesure]
            echelle = 1 if mesure == "ROC AUC" else 100
            lignes.append({"Réglages": reglage, "Modèle": nom, "Mesure": mesure,
                           "Sans": round(sans[mesure].mean() * echelle, 3 if echelle == 1 else 1),
                           "Avec": round(avec[mesure].mean() * echelle, 3 if echelle == 1 else 1),
                           "Gain": f"{gains.mean() * echelle:+.{3 if echelle == 1 else 2}f} ± {gains.std() * echelle:.{3 if echelle == 1 else 2}f}",
                           "Plis gagnés": f"{(gains > 0).sum()} sur 5", "Verdict": verdict(gains)})
table = pd.DataFrame(lignes)
display(Markdown("**Verdicts** (une colonne par modèle)"))
display(table.pivot_table(index=["Réglages", "Mesure"], columns="Modèle", values="Verdict", aggfunc="first", sort=False))
display(Markdown("**Gains** (points de pourcentage ; ROC AUC en valeur)"))
display(table.pivot_table(index=["Réglages", "Mesure"], columns="Modèle", values="Gain", aggfunc="first", sort=False))
with pd.option_context("display.max_rows", None):
    display(Markdown("**Détail**"))
    display(table.set_index(["Réglages", "Modèle", "Mesure"]))

**Comment lire** :
- **Précision au rappel de 50 à 80 %** : les deux modèles détectent la même part des défauts ; un gain positif veut dire que, pour en détecter autant, le modèle avec `PAY_1_corrigee` signale moins de bons clients.
- **Précision moyenne, ROC AUC** : la capacité à classer les défauts en tête, tous seuils confondus.
- **Règle marginale** : rappel et précision au seuil choisi par la règle (au plus 1 bon client pour 1 défaut, à la marge) ; ils se lisent ensemble.
- **Deux jeux de réglages** : une conclusion qui tient avec les deux ne dépend pas du réglage.

## 5. Ce que regardent les modèles (réglages A, modèle avec `PAY_1_corrigee`)

Importance par permutation sur chaque pli de validation, avec le modèle du pli : baisse de la précision moyenne, en points, quand une colonne ou une famille est mélangée (5 mélanges, graine fixée, moyenne sur les plis). `PAY_1` et `PAY_1_corrigee` sont mélangées chacune seule, puis **ensemble** : comme elles sont identiques pour 88 % des clients, chacune remplace l'autre quand elle est mélangée seule ; les mélanger ensemble mesure ce que les deux portent à elles deux.

In [ ]:
FAMILLES = {
    "PAY_1 seule": ["PAY_1"],
    "PAY_1_corrigee seule": ["PAY_1_corrigee"],
    "PAY_1 et PAY_1_corrigee ensemble": ["PAY_1", "PAY_1_corrigee"],
    "Codifications d'avril à août (PAY_2 à PAY_6)": PAY[1:],
    "Factures (BILL_AMTn)": [f"BILL_AMT{n}" for n in range(1, 7)],
    "Paiements (PAY_AMTn)": [f"PAY_AMT{n}" for n in range(1, 7)],
    "Plafond (LIMIT_BAL)": ["LIMIT_BAL"],
    "Démographie": ["SEX", "EDUCATION", "MARRIAGE", "AGE"],
}
generateur = np.random.default_rng(42)
X = original[COLONNES["Avec PAY_1_corrigee"]]
reglage = "A (essai_jeu_de_base)"
importance = {}
for nom in REGLAGES[reglage]:
    pertes = {f: [] for f in FAMILLES}
    for m, (_, validation) in zip(modeles_plis[(reglage, nom, "Avec PAY_1_corrigee")], plis):
        X_val, y_val = X.iloc[validation], y[validation]
        reference = average_precision_score(y_val, m.predict_proba(X_val)[:, 1])
        for famille, colonnes in FAMILLES.items():
            perte = []
            for _ in range(5):
                melange = X_val.copy()
                melange[colonnes] = melange[colonnes].to_numpy()[generateur.permutation(len(melange))]
                perte.append(reference - average_precision_score(y_val, m.predict_proba(melange)[:, 1]))
            pertes[famille].append(np.mean(perte))
    importance[nom] = {f: f"{np.mean(v) * 100:.2f} ± {np.std(v) * 100:.2f}" for f, v in pertes.items()}
    print(f"{nom} : fait", flush=True)
display(Markdown("**Perte de précision moyenne quand la colonne ou la famille est mélangée** (points ; moyenne ± écart-type sur les 5 plis)"))
display(pd.DataFrame(importance))

**Comment lire** : comparer les lignes « `PAY_1` seule » et « `PAY_1_corrigee` seule » montre laquelle le modèle regarde en premier ; la ligne « ensemble » donne ce que les deux portent à elles deux. Si la perte « ensemble » est à peine plus grande que celle de `PAY_1` seule, `PAY_1_corrigee` n'ajoute presque rien à ce que `PAY_1` dit déjà.

## 6. Conclusion

*À écrire par l'utilisateur après lecture des résultats.*